# Account - block B (test 2015-2016)
Settings: Accelerator = GPU, Internet = ON. **Save Version -> Save & Run All (Commit)**. Output: `results_B.zip`.

In [ ]:
# ---- 1. setup (needs Internet ON in the notebook settings) ----
import os, subprocess, shutil, glob, sys
def sh(c): print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout[-1500:])
sh("pip -q install cdflib tensorboard tqdm tabulate 2>&1 | tail -1")
if not os.path.exists("/kaggle/working/STORM-PhysNet-L1"):
    # pinned to the exact repo version the kit was tested against, so a later push to GitHub cannot change the results
    sh("cd /kaggle/working && git clone https://github.com/bnsama29-cloud/STORM-PhysNet-L1 && cd STORM-PhysNet-L1 && git checkout 72008c75e30c10212d2fba5b05739af82fdc097a")
sh("cd /kaggle/working/STORM-PhysNet-L1 && git rev-parse HEAD")
# the kit = the Kaggle dataset you created from storm_kaggle_kit.zip (folder name may differ: adjust KIT if needed)
cand = sorted(glob.glob("/kaggle/input/**/train_block.py", recursive=True), key=len)
assert cand, "kit not found: add your storm-kit dataset as an Input (right sidebar -> Add Input)"
KIT = os.path.dirname(cand[0]); print("KIT =", KIT)
shutil.copytree(KIT, "/kaggle/working/kit", dirs_exist_ok=True)
REPO = "/kaggle/working/STORM-PhysNet-L1"; KITW = "/kaggle/working/kit"
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# ---- 2. resume support: if you attach the previous run's output as an Input, its finished runs are copied back ----
OUT = "/kaggle/working/out_B"; os.makedirs(OUT, exist_ok=True)
for d in glob.glob("/kaggle/input/**/out_B", recursive=True):
    shutil.copytree(d, OUT, dirs_exist_ok=True); print("resumed from", d)
print(len(glob.glob(OUT + "/preds_*.npz")), "runs already done")

In [ ]:
# ---- 3. train block B (10 seeds x 7 model families). Safe to re-run: finished runs are skipped. ----
r = subprocess.run(f"cd {KITW} && python -u train_block.py --repo {REPO} --block B --out {OUT} ", shell=True)
print("exit code", r.returncode)

In [ ]:
# ---- zip the results for the combine step ----
import zipfile
with zipfile.ZipFile("/kaggle/working/results_B.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for d in "out_B".split():
        for f in glob.glob(f"/kaggle/working/{d}/**/*", recursive=True):
            if os.path.isfile(f): zf.write(f, os.path.relpath(f, "/kaggle/working"))
assert os.path.getsize("/kaggle/working/results_B.zip") > 1000, "zip is empty - check the out_ folders"
print(os.path.getsize("/kaggle/working/results_B.zip")/1e6, "MB ->", "/kaggle/working/results_B.zip")